# Applying the step counter to Capture-24

In the tutorial we trained a step counter on OxWalk (39 people, ~1 h each, every heel strike annotated from video) and measured its accuracy. Here we apply that model to a much larger, free-living dataset: **Capture-24**.

Capture-24 has no step labels.

**Where the step counts come from.** We trained the tutorial's step counter: random-forest walk detector + HMM smoothing + find_peaks peak counter) on all 39 OxWalk participants. Each Capture-24 recording was cut into 10 s windows and the model predicted a walk/non-walk flag and a step count for every window. Each window also carries the camera annotation that covers most of it.

**What this session is (and is not).** Without step labels we cannot measure accuracy. Instead you will (i) analyse behaviour (who walks, when, how) and (ii) check plausibility (do the predictions make physical sense?). Pick one research question, answer it in ~35 min.

## Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

sys.path.append("../..")  # utils.py lives in the repository root
import utils  # helper functions -- check out utils.py

pd.options.display.max_colwidth = 80
pd.options.display.width = 120
np.random.seed(42)  # for reproducibility

BLUE, GREY = '#2a78d6', '#9a9a9a'  # one highlight colour + grey for context
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e8e8e8', 'figure.dpi': 100})

WINDOW_SEC = 10                     # the model predicts steps per 10 s window
WIN_PER_HOUR = 3600 // WINDOW_SEC   # 360 windows = 1 hour
AGE_ORDER = ['18-29', '30-37', '38-52', '53+']

In [ ]:
# NOTE: The required dataset is already available at /srv/capture24.
# The following creates a symbolic link (shortcut in Windows) to it in the current directory.
# (Outside the course server we fall back to a local copy in the repository root, ../../capture24.)
if not os.path.exists("capture24"):
    if os.path.islink("capture24"):  # broken link from elsewhere -> replace it
        os.remove("capture24")
    src = "/srv/capture24" if os.path.exists("/srv/capture24") else os.path.abspath("../../capture24")
    os.symlink(src, "capture24")

anno_dict = pd.read_csv("capture24/annotation-label-dictionary.csv", index_col='annotation', dtype='string')
metadata = pd.read_csv("capture24/metadata.csv")
print(f"{len(metadata)} participants in capture24/metadata.csv; label schemes: {list(anno_dict.columns)}")

In [ ]:
# Step predictions (one row per 10 s window), stored next to this notebook
df = pd.read_csv(
    "capture24_steps_10s.csv.gz",
    parse_dates=['time'],
    dtype={'pid': 'category', 'sex': 'category', 'steps': 'int16', 'walk': 'int8',
           'annotation': 'string', 'label_walmsley': 'category',
           'label_willetts_specific': 'category', 'met': 'float32'},
)
df['age'] = pd.Categorical(df['age'], categories=AGE_ORDER, ordered=True)
daily = pd.read_csv("capture24_steps_daily.csv")  # one row per participant (same as steps_per_day(df) below)

# Other label schemes can be added from the dictionary, e.g.:
# df['label_doherty'] = df['annotation'].map(anno_dict['label:DohertySpecific2018'])

print(f"{len(df):,} windows = {len(df) / WIN_PER_HOUR:,.0f} h from {df['pid'].nunique()} participants; "
      f"{df['annotation'].notna().mean():.0%} of windows annotated")
df.head()

### Data dictionary (`df`, one row per 10 s window)

| column | type / unit | meaning |
|---|---|---|
| `pid` | category | participant id (`P001` ... `P151`) |
| `age` | ordered category | age band: 18-29, 30-37, 38-52, 53+ |
| `sex` | category | F / M |
| `time` | datetime | start of the 10 s window (local time) |
| `steps` | int, steps per 10 s | **predicted** steps in the window; always 0 when `walk == 0`. ×6 = steps/min |
| `walk` | 0 / 1 | walk detector (RF + HMM smoothing) says "walking" in this window |
| `annotation` | string | raw camera/diary annotation covering most of the window; `<NA>` = not annotated (camera off, privacy) |
| `label_walmsley` | category | coarse label: sleep / sedentary / light / moderate-vigorous (Walmsley 2020) |
| `label_willetts_specific` | category | finer label: sleep, sitting, standing, walking, household-chores, vehicle, bicycling, mixed-activity, manual-work, sports (Willetts 2018) |
| `met` | float, MET | metabolic equivalent of the annotated activity (1 MET = resting) |

`daily` has one row per participant: `hours_recorded`, `hours_annotated`, `total_steps`, `walking_minutes` and the same normalised to 24 h (`steps_per_day`, `walking_minutes_per_day`).

## Choose your research question or come up with your own

Ideas
- What does the daily step profile look like by hour of day? 
- Do steps/day differ by age band and by sex?
- Does the daily step profile  differ by age or sex? 
- Which activities contribute most of the daily steps?
- How much walking happens in short bouts (< 1 min)?
- Do steps/day correlate with annotated MVPA time or sedentary time?
- Where does the model count steps outside walking, and why?
- Why do some annotated-walking windows get 0 steps?
- What fraction of participants reach 7,000 / 10,000 steps/day? 
- Advanced idea: Does the published model agree with ours, and where do they differ? The model from Small et al. (a self-supervised ResNet walk detector) is published as the `stepcount` package ([github.com/OxWearables/stepcount](https://github.com/OxWearables/stepcount)). Run it on a few Capture-24 participants and compare its step counts with the RF predictions. You may need gpus


 **Things to keep in mind**
  - Recordings have different lengths for different
  participants, so normalise (e.g. steps per 12h/24 h of
  recording) before comparing people.
  - The groups are very small, so you can show confidence intervals.
  - The step counter can give wrong results, and we have no ground truth in Capture24 to check them against.
  - You can use the Capture-24 data with or without the annotations (about a third of the recorded time has no annotation).

## Your answer

**Research question:**

*...*

**Method (1-2 sentences):** which data, which normalisation, which summary / uncertainty estimate.

*...*

In [ ]:
# Your analysis here

**Main figure:**.

*...*

**Answer:**

*...*

**Limitations**

*...*